<a href="https://colab.research.google.com/github/Karmar1010/intro-ciencia-datos-tareas-KMG/blob/main/homework02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 6 · Tarea 02 — Mini proyecto: reporte ejecutivo de ventas

### pandas aplicado a un caso de negocio real

En esta tarea construirás, **pieza por pieza**, un sistema de reporte de ventas.
Dado el dataset de transacciones, producirás un DataFrame resumen exportable
a CSV, listo para entregar a un equipo directivo.

```
  transacciones.csv
         │
         ▼
  cargar_dataset()          ← validación y carga
         │
         ▼
  metricas_ciudad()         ← groupby + agregaciones
         │
         ▼
  agregar_porcentaje()      ← cálculo de pct_total
         │
         ▼
  agregar_top_categoria()   ← join con categoría dominante
         │
         ▼
  reporte_ejecutivo()       ← integración de todo
         │
         ▼
  exportar_reporte()        ← CSV listo para entregar
```

**Instrucciones**

1. Implementa cada función reemplazando `raise NotImplementedError`.
2. **Sigue el orden**: cada parte usa las anteriores.
3. **No cambies nombres ni parámetros**.
4. Ejecuta los tests de cada parte hasta ver ✅.

> 🧠 Este es el patrón de **descomposición**: el problema grande (reporte de ventas)
> se divide en subproblemas pequeños que ya sabemos resolver.

In [ ]:
# Identifícate (esto ayuda si entregas el notebook).
NOMBRE = "KAREN MARCELLA GIRALDO VASCO"   # ✏️ escribe tu nombre completo
print("Tarea lista para resolver. Ejecuta cada bloque de tests tras implementar.")

Tarea lista para resolver. Ejecuta cada bloque de tests tras implementar.


## Ejercicio 1 · Cargar y validar el dataset

Implementa `cargar_dataset(ruta)` que:
1. Lea el CSV desde `ruta` con `pd.read_csv`, parseando la columna
   `'fecha'` como datetime.
2. Verifique que las columnas esperadas existen:
   `['id', 'fecha', 'ciudad', 'categoria', 'metodo_pago', 'monto']`.
3. Devuelva el DataFrame si todo está bien.
4. Lanza `ValueError('columnas faltantes')` si falta alguna columna.

**Esta función es la base del proyecto**: las siguientes la usan.

In [ ]:
import pandas as pd

def cargar_dataset(ruta):
    df = pd.read_csv("/content/transacciones.csv", parse_dates=['fecha'])
    columnas_esperadas = ['id', 'fecha', 'ciudad', 'categoria', 'metodo_pago', 'monto']
    for col in columnas_esperadas:
        if col not in df.columns:
            raise ValueError('columnas faltantes')
    return df


In [ ]:
# === Tests visibles · Ejercicio 1 ===
import os
ruta = os.path.join('..', 'datasets', 'transacciones.csv')
_df = cargar_dataset(ruta)
assert _df.shape[0] == 120
assert _df.shape[1] == 6
assert str(_df['fecha'].dtype).startswith('datetime')
print("✅ Ejercicio 1: tests visibles superados.")

✅ Ejercicio 1: tests visibles superados.


In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 1 ===
# Verificar que el DataFrame tiene los tipos correctos
assert str(_df['fecha'].dtype).startswith('datetime')
assert _df['monto'].dtype.kind in ('i', 'f')
assert _df['ciudad'].dtype.kind == 'O'
print("✅ Ejercicio 1: tests adicionales superados.")

✅ Ejercicio 1: tests adicionales superados.


## Ejercicio 2 · Calcular métricas por ciudad

Implementa `metricas_ciudad(df)` que reciba el DataFrame (devuelto
por `cargar_dataset`) y devuelva un **DataFrame** con una fila por ciudad
y las siguientes columnas:
- `'ciudad'`
- `'n_transacciones'`: número de transacciones
- `'total'`: suma de montos
- `'promedio'`: promedio de monto (float)
- `'maximo'`: monto máximo

Ordenado de mayor a menor por `'total'`.

**Usa `cargar_dataset` para obtener el DataFrame en tus tests.**

In [ ]:
def metricas_ciudad(df):
    df_met = df.groupby('ciudad').agg(
        n_transacciones=('id', 'count'),
        total=('monto', 'sum'),
        promedio=('monto', 'mean'),
        maximo=('monto', 'max')
    ).reset_index()

    df_ordenado = df_met.sort_values(by='total', ascending=False)
    df_final = df_ordenado.reset_index(drop=True)
    return df_final

In [ ]:
# === Tests visibles · Ejercicio 2 ===
import os
ruta = os.path.join('..', 'datasets', 'transacciones.csv')
_df = cargar_dataset(ruta)
_met = metricas_ciudad(_df)
assert 'ciudad' in _met.columns
assert 'n_transacciones' in _met.columns
assert 'total' in _met.columns
assert len(_met) == 5
print("✅ Ejercicio 2: tests visibles superados.")

✅ Ejercicio 2: tests visibles superados.


In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 2 ===
assert _met.iloc[0]['total'] >= _met.iloc[-1]['total']
assert _met['n_transacciones'].sum() == 120
assert abs(_met['total'].sum() - _df['monto'].sum()) < 1
print("✅ Ejercicio 2: tests adicionales superados.")

✅ Ejercicio 2: tests adicionales superados.


## Ejercicio 3 · Agregar porcentaje del total

Implementa `agregar_porcentaje(metricas)` que reciba el DataFrame
devuelto por `metricas_ciudad` y devuelva una **copia** con una columna
adicional `'pct_total'`: el porcentaje de cada ciudad sobre el total
global, redondeado a 2 decimales.

**Ejemplo:** si el total global es 1000 y Bogota tiene 250,
`pct_total` de Bogota = 25.0

In [ ]:
def agregar_porcentaje(metricas):
    df2 = metricas.copy()
    df2['pct_total'] = ((df2['total'] / df2['total'].sum()) * 100).round(2)
    return df2

In [ ]:
# === Tests visibles · Ejercicio 3 ===
import os
ruta = os.path.join('..', 'datasets', 'transacciones.csv')
_df = cargar_dataset(ruta)
_met = metricas_ciudad(_df)
_met_pct = agregar_porcentaje(_met)
assert 'pct_total' in _met_pct.columns
assert abs(_met_pct['pct_total'].sum() - 100.0) < 0.1
print("✅ Ejercicio 3: tests visibles superados.")

✅ Ejercicio 3: tests visibles superados.


In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 3 ===
assert 'pct_total' not in _met.columns
assert (_met_pct['pct_total'] > 0).all()
print("✅ Ejercicio 3: tests adicionales superados.")

✅ Ejercicio 3: tests adicionales superados.


## Ejercicio 4 · Agregar la categoría más vendida por ciudad

Implementa `agregar_top_categoria(metricas, df)` que reciba el DataFrame
de métricas y el DataFrame original, y devuelva una **copia** de metricas
con una columna adicional `'top_categoria'`: el nombre de la categoría
con mayor suma de monto para cada ciudad.

**Pista:** calcula un groupby por `['ciudad', 'categoria']`, toma el
máximo por ciudad, y haz un merge con metricas.

In [ ]:
def agregar_top_categoria(metricas, df):
    df3 = metricas.copy()
    df_cat = df.groupby(['ciudad', 'categoria'])['monto'].sum().reset_index()

    idx_max = df_cat.groupby('ciudad')['monto'].idxmax()
    df_top = df_cat.loc[idx_max, ['ciudad', 'categoria']].rename(columns={'categoria': 'top_categoria'})

    df3 = df3.merge(df_top, on='ciudad', how='left')


    return df3




In [ ]:
# === Tests visibles · Ejercicio 4 ===
import os
ruta = os.path.join('..', 'datasets', 'transacciones.csv')
_df = cargar_dataset(ruta)
_met = metricas_ciudad(_df)
_met_top = agregar_top_categoria(_met, _df)
assert 'top_categoria' in _met_top.columns
assert len(_met_top) == 5
assert _met_top['top_categoria'].notna().all()
print("✅ Ejercicio 4: tests visibles superados.")

✅ Ejercicio 4: tests visibles superados.


In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 4 ===
cats_validas = _df['categoria'].unique().tolist()
assert _met_top['top_categoria'].isin(cats_validas).all()
print("✅ Ejercicio 4: tests adicionales superados.")

✅ Ejercicio 4: tests adicionales superados.


## Ejercicio 5 · Construir el reporte ejecutivo completo

Implementa `reporte_ejecutivo(df)` que **integre** las funciones
anteriores para producir el DataFrame de reporte completo:

1. Llama a `metricas_ciudad(df)` para obtener las métricas base.
2. Llama a `agregar_porcentaje(metricas)` para agregar `pct_total`.
3. Llama a `agregar_top_categoria(metricas, df)` para agregar `top_categoria`.
4. Devuelve el DataFrame final con columnas:
   `ciudad, n_transacciones, total, promedio, maximo, pct_total, top_categoria`

Este es el patrón de **integración**: ensamblar piezas que ya funcionan.

In [ ]:
def reporte_ejecutivo(df):
    m_base = metricas_ciudad(df)
    m_pct = agregar_porcentaje(m_base)
    df_completo = agregar_top_categoria(m_pct, df)

    columnas_ordenadas = [
        'ciudad', 'n_transacciones', 'total',
        'promedio', 'maximo', 'pct_total', 'top_categoria'
    ]
    df_final = df_completo[columnas_ordenadas]

    return df_final

In [ ]:
# === Tests visibles · Ejercicio 5 ===
import os
ruta = os.path.join('..', 'datasets', 'transacciones.csv')
_df = cargar_dataset(ruta)
_rep = reporte_ejecutivo(_df)
assert 'ciudad' in _rep.columns
assert 'pct_total' in _rep.columns
assert 'top_categoria' in _rep.columns
assert len(_rep) == 5
print("✅ Ejercicio 5: tests visibles superados.")

✅ Ejercicio 5: tests visibles superados.


In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 5 ===
assert abs(_rep['pct_total'].sum() - 100.0) < 0.1
assert _rep['n_transacciones'].sum() == 120
assert _rep.iloc[0]['total'] >= _rep.iloc[-1]['total']
print("✅ Ejercicio 5: tests adicionales superados.")

✅ Ejercicio 5: tests adicionales superados.


## Ejercicio 6 · Exportar el reporte a CSV

Implementa `exportar_reporte(df, ruta_salida)` que:
1. Genere el reporte ejecutivo usando `reporte_ejecutivo(df)`.
2. Redondee `promedio` a 2 decimales.
3. Exporte el resultado a `ruta_salida` como CSV (sin índice).
4. Imprima un mensaje: `'Reporte exportado: N ciudades'` donde N es el
   número de filas.
5. Devuelva el DataFrame exportado.

**Esta es la pieza final del proyecto**: produce el entregable.

In [ ]:
def exportar_reporte(df, ruta_salida):
    df_rep = reporte_ejecutivo(df)

    df_rep['promedio'] = df_rep['promedio'].round(2)

    df_rep.to_csv(ruta_salida, index=False)

    n_ciudades = len(df_rep)
    print(f'Reporte exportado: {n_ciudades} ciudades')

    return df_rep

In [ ]:
# === Tests visibles · Ejercicio 6 ===
import os, tempfile, pandas as pd
ruta = os.path.join('..', 'datasets', 'transacciones.csv')
_df = cargar_dataset(ruta)
_ruta_out = tempfile.mktemp(suffix='.csv')
_rep = exportar_reporte(_df, _ruta_out)
assert _rep is not None
assert len(_rep) == 5
print("✅ Ejercicio 6: tests visibles superados.")

Reporte exportado: 5 ciudades
✅ Ejercicio 6: tests visibles superados.


In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 6 ===
_leido = pd.read_csv(_ruta_out)
assert len(_leido) == 5
assert 'pct_total' in _leido.columns
assert 'top_categoria' in _leido.columns
os.unlink(_ruta_out)
print("✅ Ejercicio 6: tests adicionales superados.")

✅ Ejercicio 6: tests adicionales superados.


---
## ¡Proyecto terminado!

Si todos los tests pasan, has construido un pipeline completo de análisis:

| Parte | Concepto pandas |
|---|---|
| 1 Cargar y validar | `pd.read_csv` + validación de columnas |
| 2 Métricas por ciudad | `groupby` + `agg` múltiple |
| 3 Porcentaje del total | operación vectorizada sobre columna |
| 4 Top categoría | `groupby` anidado + `idxmax` + `merge` |
| 5 Reporte ejecutivo | **integración** de funciones anteriores |
| 6 Exportar a CSV | `to_csv` + retorno del DataFrame |

### Reto opcional (sin calificar)

- Añade una columna `'metodo_pago_principal'`: el método de pago más usado
  por ciudad (similar a `top_categoria` pero para `metodo_pago`).
- Modifica `exportar_reporte` para que también exporte un segundo CSV con el
  resumen por categoría.
- ¿Cómo añadirías una fila "TOTAL" al final del reporte con las sumas globales?

> 💡 Este tipo de pipeline —carga → transformación → agregación → exportación—
> es el corazón de cualquier proyecto ETL (Extract, Transform, Load) en
> ingeniería de datos.

In [ ]:
def agregar_top_categoria(metricas, df):

    df3 = metricas.copy()

    df_cat = df.groupby(['ciudad', 'categoria'])['monto'].sum().reset_index()

    idx_max = df_cat.groupby('ciudad')['monto'].idxmax()
    df_top = df_cat.loc[idx_max, ['ciudad', 'categoria']].rename(columns={'categoria': 'top_categoria'})

    df3 = df3.merge(df_top, on='ciudad', how='left')

    return df3

In [ ]:
def exportar_reporte_avanzado(df, ruta_salida_principal, ruta_salida_categorias):

    df_rep = reporte_ejecutivo(df)
    df_rep['promedio'] = df_rep['promedio'].round(2)
    df_rep.to_csv(ruta_salida_principal, index=False)


    df_categorias = df.groupby('categoria').agg(
        n_transacciones=('id', 'count'),
        total_ventas=('monto', 'sum')
    ).reset_index().sort_values(by='total_ventas', ascending=False)

    df_categorias.to_csv(ruta_salida_categorias, index=False)

    print(f'Reporte principal exportado ({len(df_rep)} ciudades) y reporte por categorías guardado.')
    return df_rep, df_categorias

In [ ]:
def agregar_fila_total(df_reporte):

    fila_total = {
        'ciudad': 'TOTAL / PROMEDIO GLOBAL',
        'n_transacciones': df_reporte['n_transacciones'].sum(),
        'total': df_reporte['total'].sum(),
        'promedio': df_reporte['promedio'].mean().round(2),
        'maximo': df_reporte['maximo'].max(),
        'pct_total': df_reporte['pct_total'].sum(),
        'top_categoria': '-'
    }


    df_con_total = pd.concat([df_reporte, pd.DataFrame([fila_total])], ignore_index=True)

    return df_con_total